# PMITC v1.1 benchmarks
Results of `benchmarks/pmitc_bench.py` (leave-one-case-out plus the time split, three label sets, seed 192), then a quick end-to-end re-run of T2 on the released files.

In [1]:
import json, sys, pandas as pd
sys.path.insert(0, '../benchmarks'); import pmitc_bench as B
res = json.load(open('../benchmarks/baselines.json'))
rows = []
for s in B.SETS:
    for t in ['T1','T2','T3','T3b']:
        for m, r in res['precise'][s][t]['summary'].items():
            rows.append(dict(label_set=s, task=t, model=m, loco_mean=r['loco_mean'], loco_sd=r['loco_sd'], n_folds=r['n_loco_folds'], time_split=r['time_split']))
pd.DataFrame(rows).pivot_table(index=['task','model'], columns='label_set', values='loco_mean')

label_set                     label_broad  label_standard  label_strict
task model                                                             
T1   account_age_heuristic         0.0264          0.0264        0.0076
     gate1_rule                    0.2382          0.2382        0.0953
     gradient_boosting_tuned       0.2649          0.3191        0.0906
     logistic_regression           0.2696          0.2563        0.1188
     prevalence                    0.0017          0.0017        0.0008
T2   account_age_heuristic         0.8522          0.8522        0.8688
     gate1_rule                    0.6801          0.6801        0.8437
     gradient_boosting_tuned       0.8600          0.8619        0.8591
     logistic_regression           0.7811          0.7749        0.7428
     prevalence                    0.5000          0.5000        0.5000
T3   gate1_rule                    0.6747          0.6747        0.6958
     gradient_boosting_tuned       0.6779          0.6909        0.9458
     logistic_regression           0.6832          0.7015        0.8917
     prevalence                    0.5000          0.5000        0.5000
T3b  gate1_rule                    0.6129          0.6129        0.4833
     gradient_boosting_tuned       0.8681          0.9564        0.9958
     logistic_regression           0.9302          0.9309        0.9292
     prevalence                    0.5000          0.5000        0.5000

In [2]:
pd.DataFrame(rows)[lambda d: d.task == 'T1']

,label_set,task,model,loco_mean,loco_sd,n_folds,time_split
0,label_strict,T1,account_age_heuristic,0.0076,0.0073,2,0.0006
1,label_strict,T1,gate1_rule,0.0953,0.0877,2,0.0054
2,label_strict,T1,gradient_boosting_tuned,0.0906,0.0063,2,0.0738
3,label_strict,T1,logistic_regression,0.1188,0.0384,2,0.0782
4,label_strict,T1,prevalence,0.0008,0.0005,2,0.0002
18,label_standard,T1,account_age_heuristic,0.0264,0.0288,5,0.0096
19,label_standard,T1,gate1_rule,0.2382,0.2036,5,0.0648
20,label_standard,T1,gradient_boosting_tuned,0.3191,0.2136,5,0.0905
21,label_standard,T1,logistic_regression,0.2563,0.1832,5,0.0912
22,label_standard,T1,prevalence,0.0017,0.0004,5,0.0029


In [3]:
# known shortcuts and leakage lists
res['known_shortcuts']

{'T2.share_on_this_ladder_market': 'controls are sampled one market at a time, so the value is always 1 for a control (excluded)',
 'T2.hours_first_buy_to_tnews': 'controls are matched on first-buy time within 24-72 h, so this cannot separate classes much (kept; check its univariate AUC)',
 'T3b.*': 'placebo markets are crypto-price and weather markets: volume shape, price path and buyer mix differ by market type, not by informed trading',
 'T3.log_usdc_*': 'twins are matched on total volume decile, so total volume is controlled only coarsely; pre-cut volume can still differ by market age',
 'T1.log_buy_usdc_pre / log_market_usdc_pre / log_n_buy_fills_pre': 'selection by the sources, not construction: sources flag large bets, so pre-cut size alone ranks positives highly (univariate AUC 0.93-0.98). A detector that learns size learns what the press notices.',
 'T2.prior_net_long_win_rate / n_prior_markets': 'partly construction: control pools exclude Orca- and Inca-flagged wallets (A6.10

In [4]:
# end-to-end re-run on the released files: T2, label_strict, logistic regression and baselines (no gradient boosting)
splits = pd.read_csv('../benchmarks/splits.csv')[['case_id','split']]
r = B.t2('../data', splits, 'precise', 'label_strict', tune=False, quick=True)
r['summary']

{'account_age_heuristic': {'loco_mean': 0.8688,
  'loco_sd': 0.1149,
  'loco_min': 0.7202,
  'loco_max': 1.0,
  'n_loco_folds': 3,
  'time_split': 0.8409},
 'gate1_rule': {'loco_mean': 0.8437,
  'loco_sd': 0.1255,
  'loco_min': 0.6928,
  'loco_max': 1.0,
  'n_loco_folds': 3,
  'time_split': 0.7038},
 'logistic_regression': {'loco_mean': 0.7428,
  'loco_sd': 0.064,
  'loco_min': 0.661,
  'loco_max': 0.8174,
  'n_loco_folds': 3,
  'time_split': 0.7382},
 'prevalence': {'loco_mean': 0.5,
  'loco_sd': 0.0,
  'loco_min': 0.5,
  'loco_max': 0.5,
  'n_loco_folds': 3,
  'time_split': 0.5}}